# Phase 1: MLP Error Analysis — Baseline + Extended + Logit Bias Sweep

## Foundational Setup & Diagnostic Analysis

This notebook implements a **multilayer perceptron** (784→256→128→10) on FashionMNIST and performs systematic error analysis.
Unlike the CNN experiments, the MLP serves as a lower-capacity baseline to identify class-confusion patterns that are architecture-independent.

| Section | Purpose |
|---------|---------|
| 1–3 | Dataset loading, MLP definition, baseline training (10 epochs) |
| 4–5 | Baseline evaluation with all metrics saved to `outputs/error_analysis/MLP/phase1_diagnostics/` |
| 6–7 | Extended training (30 epochs + CosineAnnealingLR) and evaluation |
| 8 | Post-hoc logit bias sweep for the worst class (Shirt) on the extended model |
| 9 | Summary comparison: baseline vs extended vs bias sweep |

All evaluation metrics are systematically saved as raw `.txt` files for downstream AI-driven analysis.


In [10]:
import sys, os

# Detect project root: look for src/ directory in CWD or parents
def _find_root(marker="src", max_up=3):
    p = os.path.abspath(os.getcwd())
    for _ in range(max_up + 1):
        if os.path.isdir(os.path.join(p, marker)):
            return p
        p = os.path.dirname(p)
    return os.path.abspath(os.getcwd())
PROJ_ROOT = _find_root()
sys.path.insert(0, PROJ_ROOT)

import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import torchvision.transforms as transforms

from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from src.train_utils import train_one_epoch
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores
)

OUT_DIR = os.path.join(PROJ_ROOT, 'outputs/error_analysis/MLP/phase1_diagnostics')
os.makedirs(OUT_DIR, exist_ok=True)

DATA_DIR = os.path.join(PROJ_ROOT, 'data')

print(f'PyTorch version: {torch.__version__}')
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')
print(f'PROJ_ROOT: {PROJ_ROOT}')
print(f'OUT_DIR: {OUT_DIR}')
print(f'DATA_DIR: {DATA_DIR}')


PyTorch version: 2.13.0+cu130
Using device: cuda
OUT_DIR: ../outputs/error_analysis/MLP/phase1_diagnostics


## 2. Load Dataset

FashionMNIST: 60k train / 10k test, 10 classes, 28×28 grayscale. Normalised to [-1, 1].


In [11]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_ds = __import__('torchvision').datasets.FashionMNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds = __import__('torchvision').datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)
class_names = train_ds.classes

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print(f'Training batches: {len(train_loader)}')
print(f'Test batches: {len(test_loader)}')
print(f'Classes: {class_names}')


100%|██████████| 26.4M/26.4M [00:06<00:00, 4.08MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 140kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 2.31MB/s]
100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]


Training batches: 938
Test batches: 40
Classes: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


## 3. MLP Architecture — 784→256→128→10, ReLU, Dropout 0.2

A simple 3-layer perceptron matching the architecture from `practice_1.ipynb`:
- Input: 784 (flattened 28×28)
- Hidden 1: 256 → ReLU → Dropout(0.2)
- Hidden 2: 128 → ReLU → Dropout(0.2)
- Output: 10 logits


In [12]:
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_1=256, hidden_2=128, num_classes=10, dropout=0.2):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, hidden_1)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_1, hidden_2)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(hidden_2, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x); x = self.relu1(x); x = self.drop1(x)
        x = self.fc2(x); x = self.relu2(x); x = self.drop2(x)
        return self.fc3(x)

model = MLP().to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


Parameters: 235,146


## 4. Baseline Training — 10 epochs

Training for 10 epochs with CrossEntropyLoss + Adam, matching `practice_1.ipynb`.


In [13]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 10

train_losses = []
model.train()
for epoch in range(num_epochs):
    epoch_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(epoch_loss)
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}')

with open(os.path.join(OUT_DIR, 'train_losses_10.txt'), 'w') as f:
    for loss in train_losses:
        f.write(f'{loss}\n')
print(f'Training losses saved to {OUT_DIR}/train_losses_10.txt')


Epoch [1/10], Loss: 0.5370
Epoch [2/10], Loss: 0.4089
Epoch [3/10], Loss: 0.3714
Epoch [4/10], Loss: 0.3544
Epoch [5/10], Loss: 0.3349
Epoch [6/10], Loss: 0.3221
Epoch [7/10], Loss: 0.3082
Epoch [8/10], Loss: 0.3046
Epoch [9/10], Loss: 0.2925
Epoch [10/10], Loss: 0.2856
Training losses saved to ../outputs/error_analysis/MLP/phase1_diagnostics/train_losses_10.txt


## 5. Baseline Evaluation — Comprehensive

All metrics are saved as raw numerical files in `outputs/error_analysis/MLP/phase1_diagnostics/`.


In [14]:
# --- 5a. Accuracy, Per-Class Metrics, Confusion Matrix ---
accuracy, cm, per_class = evaluate_detailed(model, test_loader, device, class_names, model_name='MLP_baseline')

# Save raw confusion matrix
cm_np = cm.cpu().numpy()
with open(os.path.join(OUT_DIR, 'confusion_matrix_10.txt'), 'w') as f:
    header = f'{"":>15}'
    for name in class_names:
        header += f'{name:>15}'
    f.write(header + '\n')
    for i in range(len(class_names)):
        row = f'{class_names[i]:>15}'
        for j in range(len(class_names)):
            row += f'{cm_np[i, j]:>15}'
        f.write(row + '\n')
print("Confusion matrix saved.")

# --- 5b. ROC-AUC and PR-AUC ---
probas, labels = get_all_probas_and_labels(model, test_loader, device, 10)
roc_scores = compute_roc_auc_scores(probas, labels, model_name='MLP_baseline')
pr_scores = compute_pr_auc_scores(probas, labels, model_name='MLP_baseline')

# Save consolidated metrics file
with open(os.path.join(OUT_DIR, 'metrics_summary_10.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')
    f.write(f'{"-"*55}\n')
    for i, name in enumerate(class_names):
        tpr = per_class[name]['TPR']
        prec = per_class[name]['Precision']
        f.write(f'{name:<15} {roc_scores[f"class_{i}"]:>10.4f} {pr_scores[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')
print("Metrics summary saved.")

# --- 5c. Misclassification Analysis ---
with open(os.path.join(OUT_DIR, 'misclassification_analysis_10.txt'), 'w') as f:
    f.write('Misclassification Analysis\n')
    f.write('=' * 70 + '\n\n')
    for c in range(len(class_names)):
        name = class_names[c]; errors = cm_np[c].sum() - cm_np[c, c]
        f.write(f'True: {name}  (errors: {errors})\n')
        f.write('-' * 50 + '\n')
        for p in np.argsort(-cm_np[c]):
            if p == c or cm_np[c, p] == 0: continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np[c, p]:>4}\n')
        f.write('\n')
print("Misclassification analysis saved.")

print(f"\nAll metrics saved to {OUT_DIR}/")


  Test Accuracy: 88.16%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8090     0.0156     0.8525
  Trouser             0.9630     0.0006     0.9948
  Pullover            0.8070     0.0218     0.8046
  Dress               0.8950     0.0141     0.8757
  Coat                0.8500     0.0261     0.7834
  Sandal              0.9060     0.0010     0.9902
  Shirt               0.6990     0.0306     0.7177
  Sneaker             0.9720     0.0143     0.8828
  Bag                 0.9640     0.0031     0.9718
  Ankle boot          0.9510     0.0044     0.9596
Confusion matrix saved.
Metrics summary saved.
Misclassification analysis saved.

All metrics saved to ../outputs/error_analysis/MLP/phase1_diagnostics/


## 6. Extended Training — 30 epochs + CosineAnnealingLR

Extended training with cosine learning rate decay to push MLP closer to convergence.


In [15]:
model_ext = MLP().to(device)
optimizer_ext = optim.Adam(model_ext.parameters(), lr=0.001)
scheduler = CosineAnnealingLR(optimizer_ext, T_max=30)
EPOCHS = 30

train_losses_30 = []
model_ext.train()
for epoch in range(EPOCHS):
    loss = train_one_epoch(model_ext, train_loader, criterion, optimizer_ext, device)
    train_losses_30.append(loss)
    scheduler.step()
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}], Loss: {loss:.4f}')

# torch.save(model_ext.state_dict(), os.path.join(OUT_DIR, 'model_weights.pth'))
with open(os.path.join(OUT_DIR, 'train_losses_30.txt'), 'w') as f:
    for l in train_losses_30: f.write(f'{l}\n')
print(f'Final loss: {train_losses_30[-1]:.4f}')


Epoch [1/30], Loss: 0.5443
Epoch [5/30], Loss: 0.3331
Epoch [10/30], Loss: 0.2725
Epoch [15/30], Loss: 0.2252
Epoch [20/30], Loss: 0.1881
Epoch [25/30], Loss: 0.1603
Epoch [30/30], Loss: 0.1497
Final loss: 0.1497


## 7. Extended Model Evaluation

Same evaluation pipeline applied to the 30-epoch model.


In [16]:
# --- 7a. Accuracy, Per-Class Metrics, Confusion Matrix ---
accuracy_e, cm_e, per_class_e = evaluate_detailed(model_ext, test_loader, device, class_names, model_name='MLP_extended')

cm_np_e = cm_e.cpu().numpy()
with open(os.path.join(OUT_DIR, 'confusion_matrix_30.txt'), 'w') as f:
    header = f'{"":>15}'
    for name in class_names:
        header += f'{name:>15}'
    f.write(header + '\n')
    for i in range(len(class_names)):
        row = f'{class_names[i]:>15}'
        for j in range(len(class_names)):
            row += f'{cm_np_e[i, j]:>15}'
        f.write(row + '\n')
print("Confusion matrix saved.")

# --- 7b. ROC-AUC and PR-AUC ---
probas_e, labels_e = get_all_probas_and_labels(model_ext, test_loader, device, 10)
roc_scores_e = compute_roc_auc_scores(probas_e, labels_e, model_name='MLP_extended')
pr_scores_e = compute_pr_auc_scores(probas_e, labels_e, model_name='MLP_extended')

with open(os.path.join(OUT_DIR, 'metrics_summary_30.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {accuracy_e:.2f}\n')
    f.write(f'Test Accuracy (fraction): {accuracy_e / 100:.4f}\n\n')
    f.write(f'Macro ROC-AUC: {roc_scores_e["macro"]:.6f}\n')
    f.write(f'Macro PR-AUC:  {pr_scores_e["macro"]:.6f}\n\n')
    f.write(f'{"Class":<15} {"ROC-AUC":>10} {"PR-AUC":>10} {"TPR":>10} {"Precision":>10}\n')
    f.write(f'{"-"*55}\n')
    for i, name in enumerate(class_names):
        tpr = per_class_e[name]['TPR']
        prec = per_class_e[name]['Precision']
        f.write(f'{name:<15} {roc_scores_e[f"class_{i}"]:>10.4f} {pr_scores_e[f"class_{i}"]:>10.4f} {tpr:>10.4f} {prec:>10.4f}\n')
print("Metrics summary saved.")

# --- 7c. Misclassification Analysis ---
with open(os.path.join(OUT_DIR, 'misclassification_analysis_30.txt'), 'w') as f:
    f.write('Misclassification Analysis\n')
    f.write('=' * 70 + '\n\n')
    for c in range(len(class_names)):
        name = class_names[c]; errors = cm_np_e[c].sum() - cm_np_e[c, c]
        f.write(f'True: {name}  (errors: {errors})\n')
        f.write('-' * 50 + '\n')
        for p in np.argsort(-cm_np_e[c]):
            if p == c or cm_np_e[c, p] == 0: continue
            f.write(f'  -> {class_names[p]:<15} count={cm_np_e[c, p]:>4}\n')
        f.write('\n')
print("Misclassification analysis saved.")


  Test Accuracy: 90.08%
  Class           TPR(Recall)        FPR  Precision
  ---------------------------------------------
  T-shirt/top         0.8550     0.0170     0.8482
  Trouser             0.9720     0.0011     0.9898
  Pullover            0.8410     0.0180     0.8385
  Dress               0.9130     0.0121     0.8933
  Coat                0.8560     0.0203     0.8239
  Sandal              0.9620     0.0022     0.9796
  Shirt               0.7070     0.0261     0.7505
  Sneaker             0.9700     0.0066     0.9427
  Bag                 0.9710     0.0028     0.9749
  Ankle boot          0.9610     0.0040     0.9639
Confusion matrix saved.
Metrics summary saved.
Misclassification analysis saved.


## 8. Logit Bias Sweep — Post-hoc Shirt Bias

Apply a post-hoc bias to the Shirt class logit on the extended (30-epoch) model.
This zero-cost intervention trades overall accuracy for Shirt recall by shifting the decision boundary.


In [17]:
SHIRT_IDX = 6  # class index for Shirt
BIASES = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0]
sweep = []

model_ext.eval()
with torch.no_grad():
    for bias in BIASES:
        all_preds, all_labels_list = [], []
        sh_tp = sh_fp = sh_fn = 0
        for inputs, lbls in test_loader:
            inputs, lbls = inputs.to(device), lbls.to(device)
            logits = model_ext(inputs)
            logits[:, SHIRT_IDX] += bias
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels_list.extend(lbls.cpu().numpy())
            for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
                if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp += 1
                if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp += 1
                if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn += 1
        acc = accuracy_score(all_labels_list, all_preds)
        sweep.append({
            'bias': bias, 'acc': round(acc * 100, 2),
            'tpr': round(sh_tp / (sh_tp + sh_fn + 1e-8), 4),
            'prec': round(sh_tp / (sh_tp + sh_fp + 1e-8), 4),
        })
        print(f'bias={bias:+.1f}  acc={acc*100:.2f}%  Shirt TPR={sh_tp/(sh_tp+sh_fn+1e-8):.4f}  Prec={sh_tp/(sh_tp+sh_fp+1e-8):.4f}')

bt = max(sweep, key=lambda r: r['acc'] + r['tpr'] * 100)
print(f'\nBest trade-off: bias={bt["bias"]:+.1f}  acc={bt["acc"]:.2f}%  TPR={bt["tpr"]:.4f}  Prec={bt["prec"]:.4f}')

with open(os.path.join(OUT_DIR, 'bias_sweep_results.txt'), 'w') as f:
    f.write(f'{"Bias":>6} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}\n' + '-' * 32 + '\n')
    for r in sweep:
        f.write(f'{r["bias"]:>+5.1f} {r["acc"]:>7.2f} {r["tpr"]:>9.4f} {r["prec"]:>10.4f}\n')
    f.write(f'\nBest trade-off: bias={bt["bias"]:+.1f}  acc={bt["acc"]:.2f}%  TPR={bt["tpr"]:.4f}  Prec={bt["prec"]:.4f}\n')


bias=-1.0  acc=89.75%  Shirt TPR=0.5810  Prec=0.8348
bias=-0.5  acc=90.03%  Shirt TPR=0.6450  Prec=0.7973
bias=+0.0  acc=90.08%  Shirt TPR=0.7070  Prec=0.7505
bias=+0.5  acc=89.89%  Shirt TPR=0.7580  Prec=0.6993
bias=+1.0  acc=89.44%  Shirt TPR=0.8060  Prec=0.6458
bias=+1.5  acc=88.86%  Shirt TPR=0.8470  Prec=0.6020
bias=+2.0  acc=88.08%  Shirt TPR=0.8830  Prec=0.5574

Best trade-off: bias=+2.0  acc=88.08%  TPR=0.8830  Prec=0.5574


## 9. Comparison: Baseline (10 epochs) vs Extended (30 epochs) vs Bias Sweep


In [18]:
print(f'{"Config":<30} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}')
print('-' * 55)
print(f'{"Baseline 10 epochs":<30} {accuracy:>7.2f} {per_class["Shirt"]["TPR"]:>9.4f} {per_class["Shirt"]["Precision"]:>10.4f}')
print(f'{"Extended 30 epochs":<30} {accuracy_e:>7.2f} {per_class_e["Shirt"]["TPR"]:>9.4f} {per_class_e["Shirt"]["Precision"]:>10.4f}')
print(f'{"Extended + bias="+str(bt["bias"])+" (best)":<30} {bt["acc"]:>7.2f} {bt["tpr"]:>9.4f} {bt["prec"]:>10.4f}')

# Identify worst class for MLP (not necessarily Shirt — different from CNN)
print('\nWorst TPR (baseline):')
sorted_tpr = sorted(class_names, key=lambda n: per_class[n]['TPR'])
for i, name in enumerate(sorted_tpr[:3]):
    print(f'  {i+1}. {name:<15} TPR={per_class[name]["TPR"]:.4f}  Prec={per_class[name]["Precision"]:.4f}')

print('\nWorst TPR (extended):')
sorted_tpr_e = sorted(class_names, key=lambda n: per_class_e[n]['TPR'])
for i, name in enumerate(sorted_tpr_e[:3]):
    print(f'  {i+1}. {name:<15} TPR={per_class_e[name]["TPR"]:.4f}  Prec={per_class_e[name]["Precision"]:.4f}')


Config                            Acc%  ShirtTPR  ShirtPrec
-------------------------------------------------------
Baseline 10 epochs               88.16    0.6990     0.7177
Extended 30 epochs               90.08    0.7070     0.7505
Extended + bias=2.0 (best)       88.08    0.8830     0.5574

Worst TPR (baseline):
  1. Shirt           TPR=0.6990  Prec=0.7177
  2. Pullover        TPR=0.8070  Prec=0.8046
  3. T-shirt/top     TPR=0.8090  Prec=0.8525

Worst TPR (extended):
  1. Shirt           TPR=0.7070  Prec=0.7505
  2. Pullover        TPR=0.8410  Prec=0.8385
  3. T-shirt/top     TPR=0.8550  Prec=0.8482


## Results saved to `outputs/error_analysis/MLP/phase1_diagnostics/`

| File | Contents |
|------|----------|
| `train_losses_10.txt` | Per-epoch training loss for baseline (10 epochs) |
| `train_losses_30.txt` | Per-epoch training loss for extended (30 epochs) |
| `metrics_summary_10.txt` | Baseline accuracy, per-class ROC-AUC, PR-AUC, TPR, Precision |
| `metrics_summary_30.txt` | Extended accuracy, per-class ROC-AUC, PR-AUC, TPR, Precision |
| `confusion_matrix_10.txt` | Raw confusion matrix for baseline |
| `confusion_matrix_30.txt` | Raw confusion matrix for extended |
| `misclassification_analysis_10.txt` | Baseline per-class error breakdown |
| `misclassification_analysis_30.txt` | Extended per-class error breakdown |
| `bias_sweep_results.txt` | Bias → Acc% / Shirt TPR / Shirt Prec table |
